## 1. Install dependencies

In [1]:
%pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    sentence-transformers \
    pinecone \
    langchain-core \
    langchain-pinecone \
    langchain-huggingface \
    fastapi \
    uvicorn \
    httpx \
    "pandas==2.2.3" \
    "opentelemetry-api==1.42.1"

## 2. Imports and GPU check

In [2]:
import os
import re
import torch

from google.colab import userdata

from transformers import (AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

import json
import secrets
import threading
import time
import subprocess
from getpass import getpass
import httpx
import uvicorn
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, Field
from IPython.display import display

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Google Colab, select "
        "Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 3. Configuration and model loading


In [3]:
MODEL_ID = "BioMistral/BioMistral-7B"

INDEX_NAME = "heart-disease-knowledge"

EMBEDDING_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

NAMESPACE = ""

TOP_K = 5
MAX_NEW_TOKENS = 250

print("BioMistral model:", MODEL_ID)
print("Pinecone index:", INDEX_NAME)
print("Embedding model:", EMBEDDING_MODEL)



quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit quantization configured.")



print("Loading BioMistral tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading BioMistral model...")

if "model" not in globals():
    model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()

print("BioMistral loaded successfully!")

BioMistral model: BioMistral/BioMistral-7B
Pinecone index: heart-disease-knowledge
Embedding model: sentence-transformers/all-MiniLM-L6-v2
4-bit quantization configured.
Loading BioMistral tokenizer...


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

BioMistral loaded successfully!


## 4. Text generation

In [4]:
def generate_biomistral_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    max_input_tokens=2500,
):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Prompt must not be empty.")

    inputs = tokenizer(prompt, return_tensors="pt",truncation=False, add_special_tokens=False,)

    input_length = inputs["input_ids"].shape[-1]

    if input_length > max_input_tokens:
        raise ValueError(
            f"Prompt contains {input_length} tokens. "
            f"Maximum allowed: {max_input_tokens}. "
            "Reduce the retrieved context."
        )

    input_device = model.get_input_embeddings().weight.device

    inputs = {key: value.to(input_device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(**inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    if not answer:
        raw_output = tokenizer.decode(generated_tokens,skip_special_tokens=False,)

        print("Warning: BioMistral generated an empty answer.")
        print("Raw generated tokens:", repr(raw_output))

    return answer

## 5. Pinecone and embeddings

In [6]:
pinecone_api_key = userdata.get("PINECONE_API_KEY")

if not pinecone_api_key:
    raise ValueError(
        "PINECONE_API_KEY is missing from Colab Secrets."
    )

pc = Pinecone(api_key=pinecone_api_key)

index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()

print("Pinecone connected.")
print("Index:", INDEX_NAME)
print("Dimensions:", stats.dimension)
print("Total vectors:", stats.total_vector_count)

assert stats.dimension == 384, (
    "Pinecone index dimension does not match MiniLM."
)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    },
)

test_vector = embeddings.embed_query(
    "What are the risk factors for heart disease?"
)

print("Embedding dimensions:", len(test_vector))

assert len(test_vector) == 384

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text",
    namespace=NAMESPACE,
)

Pinecone connected.
Index: heart-disease-knowledge
Dimensions: 384
Total vectors: 9690


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384


## 6. Retrieval and evidence cleaning

In [7]:
SOURCE_URLS = {
    "Heart Attack - Causes and Risk Factors _ NHLBI, NIH.pdf":
        "https://www.nhlbi.nih.gov/health/heart-attack/causes",
}


def clean_source_url(url):
    if not isinstance(url, str):
        return None

    url = url.strip()

    match = re.fullmatch(r"\[[^\]]+\]\((https?://[^)]+)\)", url)

    if match:
        url = match.group(1)

    return url if url.startswith(("https://", "http://")) else None

def retrieve_medical_evidence(question, top_k=TOP_K):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must not be empty.")

    documents = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k},
    ).invoke(question)

    evidence = []

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        source = metadata.get("source", "Unknown source")
        page = metadata.get("page")

        if isinstance(page, (int, float)):
            if float(page).is_integer():
                page = int(page)

        source_url = clean_source_url(
            metadata.get("source_url")
            or SOURCE_URLS.get(source)
        )

        evidence.append({
            "rank": rank,
            "source": source,
            "page": page,
            "source_url": source_url,
            "text": document.page_content,
        })

    return evidence


def clean_medical_text(text):
    if not isinstance(text, str):
        return ""

    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    text = re.sub(r"(?m)^\s*\d{1,3}\s*$", " ", text)
    text = re.sub(r"[☐☑☒❏□]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def looks_like_contents_page(text):
    if not text:
        return True

    numbered_headings = re.findall(
        r"\b\d+(?:\.\d+){2,}\.?\s+[A-Za-z]",
        text
    )

    return (
        len(numbered_headings) >= 3
        or "table of contents" in text.lower()
        or "contents ................................" in text.lower()
    )


def complete_sentences(text, max_chars=850):
    text = clean_medical_text(text)

    if not text:
        return ""

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)

    selected = []
    total = 0

    for sentence in sentences:
        sentence = sentence.strip()

        if len(sentence) < 25:
            continue

        if not sentence.endswith((".", "!", "?")):
            continue

        if total + len(sentence) > max_chars:
            break

        selected.append(sentence)
        total += len(sentence) + 1

    return " ".join(selected)


def select_relevant_evidence(retrieved, max_sources=3):
    selected = []
    seen_sources = set()

    for item in retrieved:
        title = item.get("source", "Unknown source")
        raw_text = item.get("text", "")

        cleaned = clean_medical_text(raw_text)

        if len(cleaned) < 100:
            continue

        if looks_like_contents_page(cleaned):
            continue

        passage = complete_sentences(cleaned)

        if len(passage) < 100:
            continue

        if title in seen_sources:
            continue

        selected.append({
            **item,
            "clean_text": passage,
        })

        seen_sources.add(title)

        if len(selected) >= max_sources:
            break

    return selected

**Test retrieval independently**

In [8]:
question = (
    "What are the common risk factors for heart disease?"
)

retrieved_evidence = retrieve_medical_evidence(question)

print("Question:", question)
print("Retrieved passages:", len(retrieved_evidence))

for item in retrieved_evidence:
    print("\n" + "=" * 70)
    print("Rank:", item["rank"])
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("Text:", item["text"][:700])

Question: What are the common risk factors for heart disease?
Retrieved passages: 5

Rank: 1
Source: Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf
Page: 11
Text: Section 1
11
Promises for Healthy Hearts
Carmen
The doctor told me that I 
have high blood pressure and 
diabetes. Both put me at risk for 
heart disease. Now that I know, 
I’m taking steps to lower my risk.
Controlling Your Risks
Risk factors are traits and habits that make you more likely to develop 
heart disease. Some risk factors you can do something about; others you 
can’t change. The more risk factors you have, the greater your chances of 
developing heart disease.

Rank: 2
Source: Heart attack - WHO.pdf
Page: 3
Text: A relatively small set of risk factors accounts for most of this process. Behavioural risk
factors include tobacco use, a diet high in salt, sugar and unhealthy fats, physical inactivity,
and harmful use of alcohol. Air pollution is an important environmental contribu

## 7. RAG prompts and source summaries

In [9]:
def format_evidence_context(
    evidence,
    max_chars_per_passage=850,
    max_passages=3,
):
    sections = []

    for number, item in enumerate(
        evidence[:max_passages],
        start=1,
    ):
        passage = item.get("clean_text") or complete_sentences(
            item.get("text", ""),
            max_chars=max_chars_per_passage,
        )

        if not passage:
            continue

        sections.append(
            f"[{number}] Document: {item['source']}\n"
            f"Medical evidence: {passage}"
        )

    return "\n\n".join(sections)



def build_rag_prompt(question, evidence):
    if not evidence:
        raise ValueError("No relevant medical evidence available.")

    context = format_evidence_context(evidence,max_chars_per_passage=850,
        max_passages=3,
    )

    if not context:
        raise ValueError("No usable medical evidence available.")

    return f"""<s>[INST]
You are BioMistral, a medical information assistant
supporting healthcare professionals.

Answer the doctor's question using the medical documents below.

Rules:
- Answer the specific question directly.
- Combine relevant information from the supplied documents.
- Write clear, complete paragraphs in understandable medical language.
- Explain technical terms briefly when necessary.
- Do not copy document headers, page numbers, section numbers,
  checkboxes, or broken PDF formatting.
- Do not repeat irrelevant population statistics.
- Do not invent facts, clinical recommendations, or references.
- If the evidence is insufficient, clearly say so.
- Do not present a model prediction as a confirmed diagnosis.
- Do not give unsupported patient-specific treatment instructions.
- Finish with a complete sentence.

MEDICAL DOCUMENT EVIDENCE:
{context}

DOCTOR'S QUESTION:
{question}

Write a clear medical explanation:
[/INST]"""

def generate_source_summary(question, item):
    passage = item.get("clean_text", "")

    if not passage:
        return None

    prompt = f"""<s>[INST]
You are summarizing evidence from one medical document.

Doctor's question:
{question}

Document title:
{item['source']}

Document evidence:
{passage}

Write ONE or TWO clear sentences explaining what this
document contributes to answering the doctor's question.

Requirements:
- Use only information supported by this passage.
- Use complete, understandable sentences.
- Do not copy the passage verbatim.
- Do not include page numbers, section numbers, or PDF formatting.
- Do not invent information.
- If the passage does not answer the question, reply exactly:
NOT_RELEVANT

Summary:
[/INST]"""

    summary = generate_biomistral_answer(
        prompt,
        max_new_tokens=110,
    ).strip()

    if not summary or "NOT_RELEVANT" in summary:
        return None

    if not summary.endswith((".", "!", "?")):
        return None

    return summary

**Generate a BioMistral RAG answer**

In [10]:
question = "What are the common risk factors for heart disease?"

retrieved_evidence = retrieve_medical_evidence(
    question,
    top_k=5,
)

rag_prompt = build_rag_prompt(
    question,
    retrieved_evidence,
)

print("Prompt tokens:", len(tokenizer(rag_prompt)["input_ids"]))
print("Prompt begins:", repr(rag_prompt[:30]))
print("Prompt ends:", repr(rag_prompt[-30:]))

rag_answer = generate_biomistral_answer(
    rag_prompt,
    max_new_tokens=250,
)

print("\nQUESTION:")
print(question)

print("\nBIOMISTRAL RAG ANSWER:")
print(rag_answer if rag_answer else "[Empty response]")

print("\nRETRIEVED SOURCES:")

for item in retrieved_evidence:
    page_str = f", page {item['page']}" if item.get("page") is not None else ""
    url_str = f" ({item['source_url']})" if item.get("source_url") else None

    print(f"[{item['rank']}] {item['source']}{page_str}{url_str or ''}")

Prompt tokens: 740
Prompt begins: '<s>[INST]\nYou are BioMistral, '
Prompt ends: 'r medical explanation:\n[/INST]'

QUESTION:
What are the common risk factors for heart disease?

BIOMISTRAL RAG ANSWER:
Heart disease is a complex condition that can be influenced by a variety of factors. Common risk factors for heart disease include high blood pressure, high cholesterol, diabetes, smoking, a diet high in sodium, saturated fats, and sugars, lack of physical activity, obesity, exposure to secondhand smoke, stress, and lack of sleep. These risk factors can impact and worsen each other’s effects, and having more than one risk factor increases the risk of developing heart disease.

RETRIEVED SOURCES:
[1] Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf, page 11
[2] Heart attack - WHO.pdf, page 3
[3] The_Truth_about_Hispanic_Latina_Women_and_Heart_Disease_Factsheet.pdf, page 2
[4] Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease

**Create and validate the reusable RAG function**

In [11]:
def answer_medical_question(question, top_k=5):

    retrieved = retrieve_medical_evidence(
        question,
        top_k=top_k,
    )

    if not retrieved:
        return {
            "question": question,
            "answer": "Insufficient retrieved evidence.",
            "status": "no_evidence",
            "evidence": [],
            "used_evidence": [],
        }

    used_evidence = retrieved[:3]

    prompt = build_rag_prompt(
        question,
        used_evidence,
    )

    answer = generate_biomistral_answer(
        prompt,
        max_new_tokens=300,
    )

    return {
        "question": question,
        "answer": answer,
        "status": "generated" if answer else "generation_failed",
        "evidence": retrieved,
        "used_evidence": used_evidence,
    }


result = answer_medical_question(
    "What are the common risk factors for heart disease?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES SUPPLIED TO BIOMISTRAL:")

for item in result["used_evidence"]:

    page_str = f", page {item['page']}" if item.get("page") is not None else ""

    url_str = f" ({item['source_url']})" if item.get("source_url") else ""

    print(f"[{item['rank']}] {item['source']}{page_str}{url_str}")


assert result["status"] == "generated"
assert result["answer"]
assert len(result["evidence"]) == 5
assert len(result["used_evidence"]) == 3

print("\nRAG response structure verified successfully!")

ANSWER:
Heart disease is a complex condition that can be influenced by a variety of factors. Common risk factors for heart disease include high blood pressure, high cholesterol, diabetes, smoking, a diet high in sodium, saturated fats, and sugars, lack of physical activity, obesity, exposure to secondhand smoke, stress, and lack of sleep. These risk factors can impact and worsen each other’s effects, and having more than one risk factor increases the risk of developing heart disease.

SOURCES SUPPLIED TO BIOMISTRAL:
[1] Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf, page 11
[2] Heart attack - WHO.pdf, page 3
[3] The_Truth_about_Hispanic_Latina_Women_and_Heart_Disease_Factsheet.pdf, page 2

RAG response structure verified successfully!


**Test multiple medical questions**

In [12]:
test_questions = [
    "What are the common risk factors for heart disease?",
    "Which heart disease risk factors can be modified?",
    "Which heart disease risk factors cannot be modified?",
    "How can unhealthy lifestyle habits contribute to heart disease?",
]

rag_test_results = []

for test_question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", test_question)

    result = answer_medical_question(
        test_question,
        top_k=5,
    )

    rag_test_results.append(result)

    print("\nBIOMISTRAL ANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for item in result["evidence"]:
        print(
            f"[{item['rank']}] "
            f"{item['source']}, page {item['page']}"
        )


QUESTION: What are the common risk factors for heart disease?

BIOMISTRAL ANSWER:
Heart disease is a complex condition that can be influenced by a variety of factors. Common risk factors for heart disease include high blood pressure, high cholesterol, diabetes, smoking, a diet high in sodium, saturated fats, and sugars, lack of physical activity, obesity, exposure to secondhand smoke, stress, and lack of sleep. These risk factors can impact and worsen each other’s effects, and having more than one risk factor increases the risk of developing heart disease.

SOURCES:
[1] Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf, page 11
[2] Heart attack - WHO.pdf, page 3
[3] The_Truth_about_Hispanic_Latina_Women_and_Heart_Disease_Factsheet.pdf, page 2
[4] Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf, page 12
[5] Heart Attack - Causes and Risk Factors _ NHLBI, NIH.pdf, page 4

QUESTION: Which heart disease risk factors can

## 8. Final API-ready RAG function

In [13]:
def prepare_rag_api_response(question, top_k=TOP_K):
    retrieved = retrieve_medical_evidence(
        question,
        top_k=top_k,
    )

    selected = select_relevant_evidence(
        retrieved,
        max_sources=3,
    )

    if not selected:
        return {
            "question": question,
            "answer": "Insufficient relevant medical evidence.",
            "status": "no_evidence",
            "sources": [],
        }
    prompt = build_rag_prompt(question, selected)

    answer = generate_biomistral_answer(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    ).strip()

    if not answer:
        return {
            "question": question,
            "answer": "",
            "status": "generation_failed",
            "sources": [],
        }

    sources = []

    for rank, item in enumerate(selected, start=1):
        summary = generate_source_summary(question,item,)
        if not summary:
            summary = item["clean_text"]

        sources.append({
            "rank": rank,
            "title": item["source"],
            "page": item.get("page"),
            "url": item.get("source_url"),
            "excerpt": "",
            "summary": summary,
        })

    return {
        "question": question,
        "answer": answer,
        "status": "generated",
        "sources": sources,
    }

## 9. Test RAG Response

In [14]:
result = prepare_rag_api_response("What are the common risk factors for heart disease?")
print(json.dumps(result, indent=2, ensure_ascii=False))
assert result["status"] == "generated", result["status"]
assert result["answer"].strip()
assert 1 <= len(result["sources"]) <= 3

{
  "question": "What are the common risk factors for heart disease?",
  "answer": "Heart disease is a complex condition that can be influenced by a variety of factors. Common risk factors for heart disease include high blood pressure, high cholesterol, diabetes, smoking, a diet high in sodium, saturated fats, and sugars, lack of physical activity, obesity, exposure to secondhand smoke, stress, and lack of sleep. These risk factors can impact and worsen each other’s effects, and having more than one risk factor increases the risk of developing heart disease.",
  "status": "generated",
  "sources": [
    {
      "rank": 1,
      "title": "Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf",
      "page": 11,
      "url": null,
      "excerpt": "",
      "summary": "The document titled \"Promises for Healthy Hearts\" provides information on how to prevent heart disease in Hispanic and Latino populations. It describes the risk factors for heart disease, in

## 10. API authentication

In [15]:
import os
import secrets
from getpass import getpass

api_key = getpass("Enter a strong shared API key (or press Enter to generate one): ")

if not api_key:
    api_key = secrets.token_urlsafe(48)
    print("Generated a new API key.")
    print("Store it privately before continuing:")
    print(api_key)

os.environ["BIOMISTRAL_API_KEY"] = api_key

del api_key

Enter a strong shared API key (or press Enter to generate one): ··········
Generated a new API key.
Store it privately before continuing:
HkqSEGpDx_me3YJoIR8CdKBXkckb-tHuH95alEXVgPNqxc3TChDEIB93bFsFjer2


## 11. Create FastAPI inference service

In [16]:
biomistral_app = FastAPI(
    title="BioMistral RAG Inference API",
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
)

bearer_scheme = HTTPBearer(auto_error=False)

inference_lock = threading.Lock()


class MedicalQuestionRequest(BaseModel):
    question: str = Field(min_length=5, max_length=1000)
    top_k: int = Field(default=5, ge=3, le=10)


def verify_api_key(
    credentials: HTTPAuthorizationCredentials = Depends(
        bearer_scheme
    )
):
    expected_key = os.environ["BIOMISTRAL_API_KEY"]

    if (
        credentials is None
        or not secrets.compare_digest(
            credentials.credentials,
            expected_key
        )
    ):
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )


@biomistral_app.post("/infer")
def infer(
    request: MedicalQuestionRequest,
    authorized=Depends(verify_api_key),
):

    if len(request.question.strip()) < 5:
        raise HTTPException(status_code=422, detail="Question is too short")

    if not inference_lock.acquire(blocking=False):
        raise HTTPException(
            status_code=503,
            detail="Inference service is busy"
        )

    try:
        result = prepare_rag_api_response(
            question=request.question,
            top_k=request.top_k,
        )

        if result["status"] != "generated":
            raise HTTPException(
                status_code=503,
                detail="Medical answer generation unavailable"
            )

        return result

    except HTTPException:
        raise

    except Exception:
        raise HTTPException(
            status_code=500,
            detail="Inference failed"
        )

    finally:
        inference_lock.release()


@biomistral_app.get("/health")
def health(authorized=Depends(verify_api_key)):
    return {"status": "ready"}


## 12. Start local inference server

In [17]:
def run_biomistral_server():
    uvicorn.run(
        biomistral_app,
        host="127.0.0.1",
        port=8001,
        log_level="warning",
        access_log=False,
    )

server_thread = threading.Thread(
    target=run_biomistral_server,
    daemon=True,
)

server_thread.start()

time.sleep(3)

response = httpx.get(
    "http://127.0.0.1:8001/health",
    headers={
        "Authorization": (
            f"Bearer {os.environ['BIOMISTRAL_API_KEY']}"
        )
    },
    timeout=10,
)

print("Status code:", response.status_code)
print("Response:", response.json())


Status code: 200
Response: {'status': 'ready'}


## 13. Install Cloudflare Tunnel

In [18]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared

!chmod +x /content/cloudflared

!/content/cloudflared --version

cloudflared version 2026.11.2 (built 2026-11-23-13:03 UTC)


## 14. Start temporary HTTPS tunnel


In [19]:
tunnel_log = "/content/biomistral_tunnel.log"

with open(tunnel_log, "w") as log_file:
    tunnel_process = subprocess.Popen(
        [
            "/content/cloudflared",
            "tunnel",
            "--url",
            "http://127.0.0.1:8001",
            "--no-autoupdate",
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

tunnel_url = None

for attempt in range(30):
    time.sleep(2)

    with open(tunnel_log, "r") as log_file:
        logs = log_file.read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("BioMistral API URL:", tunnel_url)
else:
    raise RuntimeError("Tunnel URL not found. Inspect the tunnel log.")


BioMistral API URL: https://sponsored-career-literary-delaware.trycloudflare.com


## 15. Test external API and authentication

In [20]:
headers = {
    "Authorization": (
        f"Bearer {os.environ['BIOMISTRAL_API_KEY']}"
    )
}

health_response = httpx.get(
    f"{tunnel_url}/health",
    headers=headers,
    timeout=20,
)

print("Authenticated health:", health_response.status_code)
print(health_response.text)

unauthorized_response = httpx.get(
    f"{tunnel_url}/health",
    timeout=20,
)

print(
    "Unauthenticated health:",
    unauthorized_response.status_code
)
assert health_response.status_code == 200
assert unauthorized_response.status_code == 401

Authenticated health: 200
{"status":"ready"}
Unauthenticated health: 401


# RAG Evaluation: Retrieval and BioMistral Answer Quality

This section evaluates the existing RAG pipeline using
manually verified medical questions and reference answers.

Evaluation criteria:
1. Retrieval relevance
2. Evidence faithfulness
3. Answer correctness
4. Citation validity
5. Response time

**Prepare evaluation dataset**

In [ ]:
import pandas as pd
from pathlib import Path

EVAL_DIR = Path("/content/rag_evaluation")
EVAL_DIR.mkdir(exist_ok=True)

questions_path = EVAL_DIR / "rag_questions.csv"

columns = [
    "question_id",
    "question",
    "reference_answer",
    "reference_source",
    "reference_page",
    "expected_behavior"
]

if not questions_path.exists():
    pd.DataFrame(columns=columns).to_csv(
        questions_path,
        index=False,
        encoding="utf-8-sig"
    )

print("Evaluation dataset:", questions_path)

evaluation_questions = pd.read_csv(
    questions_path
).fillna("")

print("Number of questions:", len(evaluation_questions))
display(evaluation_questions.head())

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

evaluation_questions = pd.read_csv(filename).fillna("")

print("Total questions:", len(evaluation_questions))
display(evaluation_questions.head())

**Run evaluation questions**

In [ ]:
if len(evaluation_questions) < 20:
    raise ValueError(
        "Prepare 20 verified questions."
    )

evaluation_results = []

for _, row in evaluation_questions.iterrows():

    question = str(row["question"]).strip()

    if not question:
        continue

    print("Evaluating:", row["question_id"])

    start_time = time.perf_counter()

    try:
        result = answer_medical_question(
            question,
            top_k=5
        )

        elapsed = time.perf_counter() - start_time

        used_evidence = result.get("used_evidence", [])

        evaluation_results.append({
            **row.to_dict(),
            "generated_answer": result.get("answer", ""),
            "status": result.get("status", ""),
            "response_time_seconds": round(elapsed, 2),
            "retrieved_count": len(result.get("evidence", [])),
            "used_evidence": json.dumps(
                used_evidence,
                ensure_ascii=False
            ),
            "retrieved_evidence": json.dumps(
                result.get("evidence", []),
                ensure_ascii=False
            )
        })

    except Exception as error:
        evaluation_results.append({
            **row.to_dict(),
            "generated_answer": "",
            "status": "error",
            "response_time_seconds": None,
            "retrieved_count": 0,
            "used_evidence": "[]",
            "retrieved_evidence": "[]",
            "error": str(error)
        })
    pd.DataFrame(evaluation_results).to_csv(
        EVAL_DIR / "rag_raw_results.csv",
        index=False,
        encoding="utf-8-sig"
    )

raw_results_df = pd.DataFrame(evaluation_results)

display(raw_results_df[
    [
        "question_id",
        "question",
        "status",
        "response_time_seconds"
    ]
])

**Check citation validity automatically**

In [ ]:
def evaluate_citations(answer, used_evidence):

    cited = {
        int(n)
        for n in re.findall(r"\[(\d+)\]", str(answer))
    }

    valid = {
        int(item["rank"])
        for item in used_evidence
    }

    invalid = cited - valid

    return {
        "citation_present": bool(cited),
        "citation_numbers_valid": len(invalid) == 0,
        "invalid_citations": sorted(invalid)
    }


citation_results = []

for _, row in raw_results_df.iterrows():

    evidence = json.loads(row["used_evidence"])

    check = evaluate_citations(
        row["generated_answer"],
        evidence
    )

    citation_results.append(check)

citation_df = pd.DataFrame(citation_results)

raw_results_df = pd.concat(
    [
        raw_results_df.reset_index(drop=True),
        citation_df
    ],
    axis=1
)

display(raw_results_df[
    [
        "question_id",
        "citation_present",
        "citation_numbers_valid",
        "invalid_citations"
    ]
])

**Manual evaluation sheet**

In [ ]:
manual_df = raw_results_df.copy()

manual_df["retrieval_relevance"] = ""
manual_df["faithfulness"] = ""
manual_df["answer_correctness"] = ""
manual_df["clinical_safety"] = ""
manual_df["reviewer_notes"] = ""

manual_path = EVAL_DIR / "rag_manual_review.csv"

manual_df.to_csv(
    manual_path,
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files

files.download(str(manual_path))

**final evaluation results**

In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd

uploaded = files.upload()

filename = next(iter(uploaded))

reviewed_df = pd.read_csv(filename)

score_columns = [
    "retrieval_relevance",
    "faithfulness",
    "answer_correctness",
    "clinical_safety"
]

for column in score_columns:
    reviewed_df[column] = pd.to_numeric(
        reviewed_df[column],
        errors="coerce"
    )

summary = []

for column in score_columns:

    scores = reviewed_df[column].dropna()

    if not scores.empty:
        summary.append({
            "Metric": column,
            "Reviewed Questions": len(scores),
            "Average Score": round(scores.mean(), 2),
            "Percentage of Maximum": round(
                scores.mean() / 2 * 100, 2
            )
        })

summary_df = pd.DataFrame(summary)

print("RAG Evaluation Summary")
display(summary_df)


EVAL_DIR = Path("/content/rag_evaluation")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(
    EVAL_DIR / "rag_evaluation_summary.csv",
    index=False
)

print("Evaluation summary saved successfully!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.bar(
    summary_df["Metric"],
    summary_df["Percentage of Maximum"]
)

plt.ylim(0, 100)
plt.ylabel("Percentage of Maximum Score (%)")
plt.xlabel("Evaluation Metric")
plt.title("BioMistral RAG Evaluation Results")

plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plt.savefig(
    EVAL_DIR / "rag_evaluation_chart.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()